# C. Co-optimising battery, hydrogen and EV flexibility

Start from A's network, including the water input. Add hydrogen reconversion, allow investment, then compare fixed EV charging with smart charging and V2G.

Run the cells from top to bottom. The main examples use one node and 24 hours of synthetic data.
An optional extension at the end uses a full synthetic year to compare daily and seasonal storage.
Source links refer to [PyPSA-Earth](https://github.com/pypsa-meets-earth/pypsa-earth/tree/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f).

## Set up the notebook

In a fresh Colab CPU session, uncomment and run the installation cell before importing packages.
If you have already imported numerical packages, restart the runtime after installation.
A local environment with `requirements-colab.txt` can skip this cell.

In [ ]:
# %pip install -q pypsa==1.2.4 linopy==0.8.0 highspy==1.15.1 "numpy<2.5" pandas==2.2.3 "plotly>=5.24.1,<6" PyYAML==6.0.3

In [ ]:
from pathlib import Path

import pypsa
import pandas as pd
import plotly.express as px
import numpy as np
import yaml

pd.options.plotting.backend = "plotly"
px.defaults.template = "plotly_white"

SOLVER = "highs"
pypsa.options.params.optimize.include_objective_constant = False
pypsa.options.params.statistics.nice_names = False

The next cell locates `tutorial-data/` or downloads it from the course repository.
To use a local copy in Colab, upload `tutorial-data.zip` before running the cell.

In [ ]:
data_dir = Path("tutorial-data")
if not data_dir.exists() and Path("../tutorial-data").exists():
    data_dir = Path("../tutorial-data")
if not data_dir.exists():
    import zipfile
    from urllib.request import urlretrieve

    archive_path = Path("tutorial-data.zip")
    if not archive_path.exists() and Path("../tutorial-data.zip").exists():
        archive_path = Path("../tutorial-data.zip")
    if not archive_path.exists():
        course_ref = "1f25795897e14d1307003110a6b6587f21155278"
        url = f"https://raw.githubusercontent.com/ljansen-iee/pypsa-meets-earth-lab-2026/{course_ref}/tutorial-data.zip"
        urlretrieve(url, archive_path)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(".")

config = yaml.safe_load((data_dir / "config/config.DEMO_BASE.yaml").read_text())

data_dir = data_dir.resolve()
work_dir = data_dir.parent / "work"

## 1. Read the prepared network and costs

Loading a prepared network follows [the sector preparation entry point](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/prepare_sector_network.py#L3832-L3868).
Here it is the synthetic starter bundled with the course. Solar, wind and OCGT remain a
small electricity system; the full workflow also converts thermal supply to explicit fuel-input Links.

Processed costs follow [the processed cost tables](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/process_cost_data.py#L8-L16)
and [scaling to the model period](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/process_cost_data.py#L613-L660).
The teaching cost files already use `Nyears = 24 / 8760`. Their values are illustrative.
The model has 24 one-hour weights, so its objective covers this day, including a day's share of annualised investment.

In [ ]:
starter_source = "bundled"  # Select "work" to use the network saved in A.
starter_dir = work_dir if starter_source == "work" else data_dir
n = pypsa.Network(starter_dir / "networks/elec_s_1_ec_lcopt_1h_sector.nc")

# Carrier colours are plotting metadata, shared with notebook A.
n.carriers.update(pd.DataFrame({"color": {
    "AC": "#a6bbc8",
    "solar": "#fdb913",
    "onwind": "#005b7f",
    "OCGT": "#d3c7ae",
    "industry electricity": "#a6bbc8",
    "battery": "#836bad",
    "H2": "#179c7d",
    "H2 Electrolysis": "#179c7d",
    "H2 export": "#4CC2A6",
    "H2 Store Tank": "#179c7d",
    "water": "#39c1cd",
    "water supply": "#39c1cd",
}}))
elec_costs = pd.read_csv(data_dir / "resources/costs_2035_elec.csv", index_col=0)
costs = pd.read_csv(data_dir / "resources/costs_2035_sec.csv", index_col=0)
nodes = n.buses.index[n.buses.carrier == "AC"]
options = config["sector"]
n.links[["bus0", "bus1", "carrier", "efficiency"]]

INFO:pypsa.network.io:New version 1.4.0 available! (Current: 1.2.4)


INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, links, loads, storage_units, stores


,bus0,bus1,carrier,efficiency
name,,,,
ZA0 H2 export,ZA0 H2,H2 export bus,,1.0
ZA0 H2 Electrolysis,ZA0,ZA0 H2,H2 Electrolysis,0.7


## 2. Add the hydrogen-to-electricity Link

Follow [H2 Fuel Cell](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/prepare_sector_network.py#L685-L697).
Its `p_nom` is input MW_H2. The cost table is per output MW_el, so the source multiplies
the fixed cost by efficiency when assigning the Link's input-side cost.

### Add colours for the new carriers

The starter already contains A's colours. These additional Carrier entries provide plotting metadata for reconversion and EVs.

In [ ]:
n.add(
    "Carrier",
    ["H2 Fuel Cell", "Li ion", "land transport EV", "BEV charger", "V2G", "EV battery storage"],
    color=["#179c7d", "#a8508c", "#a8508c", "#a8508c", "#a8508c", "#a8508c"],
);

In [ ]:
n.add("Link", nodes + " H2 Fuel Cell",
      bus0=nodes + " H2",
      bus1=nodes,
      carrier="H2 Fuel Cell",
      p_nom_extendable=True,
      efficiency=costs.at["fuel cell", "efficiency"],
      capital_cost=costs.at["fuel cell", "fixed"] * costs.at["fuel cell", "efficiency"],
      marginal_cost=costs.at["fuel cell", "marginal_cost"])
n.links.loc[nodes + " H2 Fuel Cell", ["bus0", "bus1", "efficiency", "capital_cost"]]

,bus0,bus1,efficiency,capital_cost
name,,,,
ZA0 H2 Fuel Cell,ZA0 H2,ZA0,0.5,123.287671


## 3. Allow investment in supply and storage

The source uses extendable capacities in
[renewable generation](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/add_electricity.py#L718-L733),
[battery StorageUnits](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/add_extra_components.py#L248-L263),
[electrolysis](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/prepare_sector_network.py#L660-L683) and
[H2 tanks](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/prepare_sector_network.py#L789-L802).

This teaching case is greenfield: the fixed capacities from A become investment options
with a zero minimum. Battery energy remains `p_nom × max_hours`; tank energy is independent.
The export transfer Link is also extendable, following `add_export.py`.

In [ ]:
electricity_generators = n.generators.index[n.generators.bus.isin(nodes)]
n.generators.loc[electricity_generators, "p_nom"] = 0.0
n.generators.loc[electricity_generators, "p_nom_min"] = 0.0
n.generators.loc[electricity_generators, "p_nom_extendable"] = True
# Keep the water-supply boundary from A fixed, in m3/h rather than MW_el.
n.storage_units["p_nom"] = 0.0
n.storage_units["p_nom_extendable"] = True
n.links["p_nom"] = 0.0
n.links["p_nom_extendable"] = True
n.stores["e_nom"] = 0.0
n.stores["e_nom_extendable"] = True
# Keep the supply model before adding EVs for the optional seasonal comparison.
n_supply = n.copy()
n.storage_units[["carrier", "p_nom_extendable", "max_hours", "capital_cost"]]

,carrier,p_nom_extendable,max_hours,capital_cost
name,,,,
ZA0 battery,battery,True,6.0,265.753425


## 4. Read the EV input CSVs

The input filenames follow [prepare_transport_data](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/Snakefile#L1467-L1497):
`transport_s_1_2035.csv`, `avail_profile_s_1_2035.csv`, `dsm_profile_s_1_2035.csv`
and `nodal_transport_data_s_1_2035.csv`.
The contents here are synthetic 24-hour profiles; the full workflow produces them from
transport, temperature, traffic and population inputs.

In [ ]:
transport = pd.read_csv(data_dir / "resources/demand/transport_s_1_2035.csv", index_col=0, parse_dates=True)
avail_profile = pd.read_csv(data_dir / "resources/pattern_profiles/avail_profile_s_1_2035.csv", index_col=0, parse_dates=True)
dsm_profile = pd.read_csv(data_dir / "resources/pattern_profiles/dsm_profile_s_1_2035.csv", index_col=0, parse_dates=True)
nodal_transport_data = pd.read_csv(data_dir / "resources/demand/nodal_transport_data_s_1_2035.csv", index_col=0)
pd.concat({"transport": transport, "availability": avail_profile, "minimum SOC fraction": dsm_profile}, axis=1)

,transport,availability,minimum SOC fraction
,ZA0,ZA0,ZA0
snapshot,,,
2013-01-01 00:00:00,0.0,0.8,0.00
2013-01-01 01:00:00,0.0,0.8,0.00
2013-01-01 02:00:00,0.0,0.8,0.00
2013-01-01 03:00:00,0.0,0.8,0.00
2013-01-01 04:00:00,0.0,0.8,0.00
2013-01-01 05:00:00,0.0,0.8,0.00
2013-01-01 06:00:00,0.0,0.8,0.00
2013-01-01 07:00:00,0.2,0.2,0.75


## 5. Build the baseline without EV flexibility

The baseline uses `bev_dsm: false` and `v2g: false`. Add the EV bus, driving-energy Load
and BEV charger, following the [public workflow](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/prepare_sector_network.py#L2354-L2406).
There is no EV Store or V2G Link in this case, so charging follows the demand profile.
The fleet, driving service, charger power and connection profile stay fixed across cases.

The synthetic availability profile retains 20% of the aggregate fleet connected during
commuting hours and 80% in other hours. That keeps the baseline feasible without an
EV Store. Connection availability is separate from the participating inventory parameter.


In [ ]:
electric_share = options["land_transport_electric_share"]

n.add("Bus", nodes + " EV battery", location=nodes, carrier="Li ion")

# Smooth driving demand over three hours, wrapping around this cyclic teaching day.
transport_shift_1 = pd.DataFrame(np.roll(transport.to_numpy(), 1, axis=0), index=transport.index, columns=transport.columns)
transport_shift_2 = pd.DataFrame(np.roll(transport.to_numpy(), 2, axis=0), index=transport.index, columns=transport.columns)
p_set = electric_share * (transport + transport_shift_1 + transport_shift_2) / 3

n.add("Load", nodes, suffix=" land transport EV",
      bus=nodes + " EV battery",
      carrier="land transport EV",
      p_set=p_set.loc[n.snapshots])

In [ ]:
# Cars × MW/car gives fleet charger input power; availability then limits each hour.
p_nom = nodal_transport_data["number cars"] * options["bev_charge_rate"] * electric_share
n.add("Link", nodes,
      overwrite=True,
      suffix=" BEV charger",
      bus0=nodes,
      bus1=nodes + " EV battery",
      carrier="BEV charger",
      p_nom=p_nom,
      efficiency=options["bev_charge_efficiency"],
      p_max_pu=avail_profile.loc[n.snapshots])
n_base = n.copy()
# With DSM and V2G off, no EV inventory or reverse Link is added.
n.links.loc[nodes + " BEV charger", ["p_nom", "efficiency"]]

,p_nom,efficiency
name,,
ZA0 BEV charger,1.1,0.9


## 6. Enable smart charging and V2G in EV75

The flexibility case uses `bev_dsm: true`, `v2g: true` and `bev_availability: 0.75`.
The public code adds [V2G when enabled](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/prepare_sector_network.py#L2408-L2419)
and an [EV battery Store when DSM is enabled](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/prepare_sector_network.py#L2421-L2439).

For 100 cars at 0.05 MWh/car and 75% participation, the flexible inventory is 3.75 MWh.
Its minimum-SOC profile reserves energy for mobility. The baseline has no participating
EV inventory; `bev_availability` is unused and DSM is disabled.


In [ ]:
config_ev75 = yaml.safe_load((data_dir / "config/config.DEMO_EV75.yaml").read_text())
flex_options = config_ev75["sector"]

if flex_options["v2g"]:
    n.add("Link", nodes,
          suffix=" V2G",
          overwrite=True,
          bus0=nodes + " EV battery",
          bus1=nodes,
          carrier="V2G",
          p_nom=p_nom,
          efficiency=options["bev_charge_efficiency"],
          p_max_pu=avail_profile.loc[n.snapshots])

if flex_options["bev_dsm"]:
    # Cars × MWh/car × participation gives usable EV inventory, independently of charger MW.
    e_nom = (nodal_transport_data["number cars"] * options["bev_energy"]
             * flex_options["bev_availability"] * electric_share)
    n.add("Store", nodes,
          suffix=" EV battery storage",
          overwrite=True,
          bus=nodes + " EV battery",
          carrier="EV battery storage",
          e_cyclic=True,
          e_nom=e_nom,
          e_min_pu=dsm_profile.loc[n.snapshots],
          e_max_pu=1.0)

n_ev75 = n.copy()
n_ev75.stores[["carrier", "e_nom", "e_nom_extendable", "e_cyclic"]]

,carrier,e_nom,e_nom_extendable,e_cyclic
name,,,,
ZA0 H2 Store Tank,H2 Store Tank,0.00,True,True
ZA0 EV battery storage,EV battery storage,3.75,False,True


## 7. Compare three scenarios

`DEMO_BASE` has no EV demand shifting or V2G and retains 1 MW_H2 of constant export.
`DEMO_EV75` enables both smart charging and V2G with 75% participating EV inventory,
keeping the fleet, driving demand, charger power and connection profile unchanged.
`DEMO_EXPORT2` doubles hydrogen export while retaining the baseline's EV flexibility off.
The stationary electricity battery remains available in every case.


In [ ]:
n_export2 = n_base.copy()
config_export2 = yaml.safe_load((data_dir / "config/config.DEMO_EXPORT2.yaml").read_text())
n_export2.loads_t.p_set["H2 export load"] *= (
    config_export2["export"]["h2export"] / config["export"]["h2export"])

pd.DataFrame({
    "number cars": [nodal_transport_data["number cars"].sum()] * 3,
    "bev_dsm": [options["bev_dsm"], flex_options["bev_dsm"], config_export2["sector"]["bev_dsm"]],
    "v2g": [options["v2g"], flex_options["v2g"], config_export2["sector"]["v2g"]],
    "participating EV inventory [MWh_el]": [0.0, e_nom.sum(), 0.0],
    "H2 delivery [MWh_H2/day]": [
        network.loads_t.p_set["H2 export load"].mul(network.snapshot_weightings.generators).sum()
        for network in [n_base, n_ev75, n_export2]],
}, index=["BASE", "EV75", "EXPORT2"])

,number cars,bev_dsm,v2g,participating EV inventory [MWh_el],H2 delivery [MWh_H2/day]
BASE,100.0,False,False,0.00,24.0
EV75,100.0,True,True,3.75,24.0
EXPORT2,100.0,False,False,0.00,48.0


In [ ]:
n_base.optimize(solver_name=SOLVER, threads=1, log_to_console=False);
# Expect an optimal combination of renewable capacity, battery power and H2 inventory.
# A fuel cell may receive zero capacity if other options are cheaper for this day.

INFO:linopy.model: Solve problem using Highs solver


INFO:linopy.model:Solver options:
 - threads: 1
 - log_to_console: False


INFO:linopy.io: Writing time: 0.31s


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 320 primals, 752 duals
Objective: 7.74e+03
Solver: highs
Runtime: 0.01s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-ext-p-lower, Generator-ext-p-upper, Link-fix-p-lower, Link-fix-p-upper, Link-ext-p-lower, Link-ext-p-upper, Store-ext-e-lower, Store-ext-e-upper, StorageUnit-ext-p_dispatch-lower, StorageUnit-ext-p_dispatch-upper, StorageUnit-ext-p_store-lower, StorageUnit-ext-p_store-upper, StorageUnit-ext-state_of_charge-lower, StorageUnit-ext-state_of_charge-upper, StorageUnit-energy_balance, Store-energy_balance were not assigned to the network.


In [ ]:
n_ev75.optimize(solver_name=SOLVER, threads=1, log_to_console=False);
# Smart charging and V2G add flexibility at the same mobility service.
# Charging can move to favourable hours, and the EV Store can support the electricity bus.

INFO:linopy.model: Solve problem using Highs solver


INFO:linopy.model:Solver options:
 - threads: 1
 - log_to_console: False


INFO:linopy.io: Writing time: 0.29s


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 392 primals, 872 duals
Objective: 7.65e+03
Solver: highs
Runtime: 0.01s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-ext-p-lower, Generator-ext-p-upper, Link-fix-p-lower, Link-fix-p-upper, Link-ext-p-lower, Link-ext-p-upper, Store-fix-e-lower, Store-fix-e-upper, Store-ext-e-lower, Store-ext-e-upper, StorageUnit-ext-p_dispatch-lower, StorageUnit-ext-p_dispatch-upper, StorageUnit-ext-p_store-lower, StorageUnit-ext-p_store-upper, StorageUnit-ext-state_of_charge-lower, StorageUnit-ext-state_of_charge-upper, StorageUnit-energy_balance, Store-energy_balance were not assigned to the network.


In [ ]:
n_export2.optimize(solver_name=SOLVER, threads=1, log_to_console=False);
# More hydrogen export adds a service requirement; compare its supply response, not only total cost.

INFO:linopy.model: Solve problem using Highs solver


INFO:linopy.model:Solver options:
 - threads: 1
 - log_to_console: False


INFO:linopy.io: Writing time: 0.28s


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 320 primals, 752 duals
Objective: 9.07e+03
Solver: highs
Runtime: 0.01s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-ext-p-lower, Generator-ext-p-upper, Link-fix-p-lower, Link-fix-p-upper, Link-ext-p-lower, Link-ext-p-upper, Store-ext-e-lower, Store-ext-e-upper, StorageUnit-ext-p_dispatch-lower, StorageUnit-ext-p_dispatch-upper, StorageUnit-ext-p_store-lower, StorageUnit-ext-p_store-upper, StorageUnit-ext-state_of_charge-lower, StorageUnit-ext-state_of_charge-upper, StorageUnit-energy_balance, Store-energy_balance were not assigned to the network.


### Read power, inventory and operation separately

The first chart compares electricity supply/discharge power in MW_el. The next
table reports electrolyser input in MW_el and fuel-cell input in MW_H2; a zero
fuel-cell row is retained to show an available option receiving no investment.
Battery energy below is power × six hours, while Store energy is independently
sized. Read those capacities together with the EV75 inventory trajectory.

In [ ]:
cases = {"DEMO_BASE": n_base, "DEMO_EV75": n_ev75, "DEMO_EXPORT2": n_export2}
capacity_cases = pypsa.NetworkCollection(pd.Series(cases).rename_axis("scenario"))
power_capacity = (
    capacity_cases.statistics.optimal_capacity(
        components=["Generator", "Link", "StorageUnit"], bus_carrier="AC",
        groupby="carrier", aggregate_across_components=True, nice_names=False,
    ).clip(lower=0).unstack("scenario").fillna(0.0).round(2)
    .loc[lambda x: x.ne(0).any(axis=1)]
)
fig = power_capacity.T.plot.bar(
    barmode="group", color_discrete_map=n.carriers.color.to_dict(), height=380)
fig.update_yaxes(title_text="Electricity supply/discharge capacity [MW_el]")
fig.show()

In [ ]:
# Link ratings refer to bus0: MW_el for electrolysis, MW_H2 for the fuel cell.
conversion_capacity = capacity_cases.statistics.optimal_capacity(
    components=["Link"], carrier=["H2 Electrolysis", "H2 Fuel Cell"], at_port="bus0",
    groupby="carrier", aggregate_across_components=True, nice_names=False, drop_zero=False,
).unstack("scenario").fillna(0.0).round(2)
conversion_capacity

scenario,DEMO_BASE,DEMO_EV75,DEMO_EXPORT2
carrier,,,
H2 Electrolysis,3.75,3.79,7.27
H2 Fuel Cell,0.00,0.00,0.00


In [ ]:
# StorageUnit energy capacity is nominal power × max_hours in every scenario.
pd.DataFrame({
    "BASE": n_base.storage_units.eval("p_nom_opt * max_hours"),
    "EV75": n_ev75.storage_units.eval("p_nom_opt * max_hours"),
    "EXPORT2": n_export2.storage_units.eval("p_nom_opt * max_hours"),
}).rename_axis("Stationary battery inventory capacity [MWh_el]").round(2)

,BASE,EV75,EXPORT2
Stationary battery inventory capacity [MWh_el],,,
ZA0 battery,82.17,81.08,82.17


In [ ]:
pd.DataFrame(
    {"BASE": n_base.stores.e_nom_opt,
    "EV75": n_ev75.stores.e_nom_opt,
    "EXPORT2": n_export2.stores.e_nom_opt}
).fillna(0.0).rename_axis("Carrier-specific energy inventory [MWh]").round(2)

,BASE,EV75,EXPORT2
Carrier-specific energy inventory [MWh],,,
ZA0 EV battery storage,0.00,3.75,0.00
ZA0 H2 Store Tank,14.64,14.88,27.79


In [ ]:
fig = pd.DataFrame({
    "stationary battery [MWh_el]": n_ev75.storage_units_t.state_of_charge["ZA0 battery"],
    "H2 tank [MWh_H2]": n_ev75.stores_t.e["ZA0 H2 Store Tank"],
    "EV inventory [MWh_el]": n_ev75.stores_t.e["ZA0 EV battery storage"],
}).plot(
    height=400, labels={"value": "EV75 inventory [carrier-labelled MWh]"},
    color_discrete_sequence=n.carriers.loc[
        ["battery", "H2 Store Tank", "EV battery storage"], "color"].to_list())
fig.show()

## 8. Save the networks

The directory and filename pattern follows
[the overnight solve rule](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/Snakefile#L1873-L1910).
Save these networks and their configurations under `work/` to retain your experiments.
Part D applies the same analysis patterns to the completed full-year country networks.

The full model uses [extra_functionality](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/solve_network.py#L1083-L1160)
and [solve_network](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/solve_network.py#L1162-L1205); use its workflow solve
rule for real prepared networks so the configured constraints are applied.

In [ ]:
case_configs = {"DEMO_BASE": config, "DEMO_EV75": config_ev75, "DEMO_EXPORT2": config_export2}
filename = "elec_s_1_ec_lcopt_1h_1h_2035_0.071.nc"
for name, network in cases.items():
    network.meta = {"tutorial": "Synthetic 24-hour notebook result; not a full ZA run",
                    "scenario": name, "starter_source": starter_source,
                    "scenario_config": case_configs[name],
                    "source_commit": "d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f",
                    "water_l_per_kg_h2": 20.0, "h2_mwh_per_kg": 33.3333 / 1000}
    result_dir = work_dir / "results" / name / "postnetworks"
    result_dir.mkdir(parents=True, exist_ok=True)
    network.export_to_netcdf(result_dir / filename)
    config_dir = work_dir / "config"
    config_dir.mkdir(parents=True, exist_ok=True)
    (config_dir / f"config.{name}.yaml").write_text(yaml.safe_dump(case_configs[name]), encoding="utf-8")

INFO:pypsa.network.io:Exported network 'Unnamed Network' saved to 'C:\Users\ljansen\projects\pypsa-meets-earth-lab-2026\tmp\final-review-execution\work\results\DEMO_BASE\postnetworks\elec_s_1_ec_lcopt_1h_1h_2035_0.071.nc contains: buses, carriers, links, loads, storage_units, generators, sub_networks, stores


INFO:pypsa.network.io:Exported network 'Unnamed Network' saved to 'C:\Users\ljansen\projects\pypsa-meets-earth-lab-2026\tmp\final-review-execution\work\results\DEMO_EV75\postnetworks\elec_s_1_ec_lcopt_1h_1h_2035_0.071.nc contains: buses, carriers, links, loads, storage_units, generators, sub_networks, stores


INFO:pypsa.network.io:Exported network 'Unnamed Network' saved to 'C:\Users\ljansen\projects\pypsa-meets-earth-lab-2026\tmp\final-review-execution\work\results\DEMO_EXPORT2\postnetworks\elec_s_1_ec_lcopt_1h_1h_2035_0.071.nc contains: buses, carriers, links, loads, storage_units, generators, sub_networks, stores


## Interpretation and recap

Discuss these questions using the tables and plots above. Use them for the joint A–C recap and quiz after the break.

1. At fixed mobility and H2 service, does EV flexibility substitute for the stationary battery, H2 inventory or neither in these results? Use the capacity tables and operation to explain the response.
2. Which investments change when `DEMO_EXPORT2` doubles H2 delivery? Explain why this comparison changes service and why the capacity response need not be proportional.
3. Why can an available fuel-cell technology have zero optimal capacity? Which assumption would you vary to test that explanation?

**Later reflection:** What would a longer chronological weather period add to your storage conclusion? Revisit this in D and, after the core, in the optional seasonal extension.


## 9. Optional: from a day to a seasonal hydrogen cycle

Compare two full-year cases with the same electricity demand, hydrogen export and
annual solar yield. `YEAR_FLAT` repeats a daily solar shape. `YEAR_SEASONAL` adds
a smooth seasonal envelope, strongest in January and weakest in July. These are
synthetic Southern Hemisphere profiles, not South African weather observations.

Use three-hour snapshots for a shorter solve: 2,920 consecutive intervals represent
8,760 hours. Set `step_hours = 1` for the hourly version. All snapshot weightings,
including `stores`, use the interval duration. Storage remains linked across every
interval and the year boundary; it is not reset each day or month.

Start from the supply model saved before EVs were added. For this experiment,
solar supplies all electricity: wind and OCGT have zero investment potential.
The six-hour stationary battery, electrolysis, fuel cell, tank, export and water
input remain. This isolates the interaction of daily electricity storage and
seasonal hydrogen storage.

In [ ]:
step_hours = 3
snapshots = pd.date_range("2013-01-01", "2014-01-01", inclusive="left", freq=f"{step_hours}h")
hour = snapshots.hour.to_numpy()
day = snapshots.dayofyear.to_numpy()
daily_solar = np.maximum(0.0, np.sin(np.pi * (hour - 6) / 12))
season = np.cos(2 * np.pi * (day - 15) / 365)

solar_profiles = pd.DataFrame({
    "YEAR_FLAT": 0.6 * daily_solar,
    "YEAR_SEASONAL": 0.6 * daily_solar * (1 + 0.65 * season),
}, index=snapshots)
solar_profiles.mean().rename("Annual mean solar capacity factor")

YEAR_FLAT        0.181066
YEAR_SEASONAL    0.181066
Name: Annual mean solar capacity factor, dtype: float64

In [ ]:
fig = solar_profiles.resample("MS").mean().plot(
    color_discrete_sequence=[n.carriers.at["solar", "color"]] * 2,
    line_dash="variable", line_dash_sequence=["dash", "solid"],
    height=300, labels={"value": "Monthly mean solar capacity factor"})
fig.show()

### Annual costs and an underground hydrogen Store

The earlier cost tables contain one day's share of annualised investment. Restore
annual costs once, because these cases represent a whole year. Marginal costs
remain per MWh and are multiplied by duration in the objective.

Add `ZA0 H2 UHS`, following the public builder's
[underground Store pattern](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/prepare_sector_network.py#L755-L787).
Assume a hypothetical 20 GWh site at **10 EUR/MWh_H2/year**, with no standing losses.
Both the potential and cost are illustrative; this does not establish cavern
availability in South Africa. The existing tank retains its higher cost.
The new Carrier entry supplies a plotting colour.

Cheap energy inventory makes storage across months competitive with building
enough solar to meet the weakest season directly. More time coverage alone need
not produce seasonal storage if backup, wind or additional solar is cheaper.

In [ ]:
n_year = n_supply.copy()
n_year.set_snapshots(snapshots)
# Duration weights affect both operating costs and the storage energy recursion.
n_year.snapshot_weightings.loc[:, :] = step_hours

# Restore annual capital costs once; marginal costs remain per MWh.
Nyears = n_supply.snapshot_weightings.objective.sum() / 8760
for component in (n_year.generators, n_year.links, n_year.storage_units, n_year.stores):
    component["capital_cost"] /= Nyears

n_year.generators.loc[["ZA0 onwind", "ZA0 OCGT"], "p_nom_max"] = 0.0
n_year.generators_t.p_max_pu = pd.DataFrame({
    "ZA0 solar": solar_profiles["YEAR_FLAT"], "ZA0 onwind": 0.3,
}, index=snapshots)
n_year.loads_t.p_set = pd.DataFrame({
    "ZA0": 4 + np.sin(2 * np.pi * (hour - 16) / 24),
    "H2 export load": 1.0,
}, index=snapshots)
# The constant industry electricity Load and 10 m3/h water boundary remain from A.
n_year.add("Carrier", "H2 UHS", color="#179c7d")
n_year.add("Store", nodes + " H2 UHS",
      bus=nodes + " H2",
      carrier="H2 UHS",
      e_nom_extendable=True,
      e_nom_max=20_000,
      e_cyclic=True,
      capital_cost=10.0)
n_year.stores[["carrier", "capital_cost", "e_nom_max", "e_cyclic"]]

,carrier,capital_cost,e_nom_max,e_cyclic
name,,,,
ZA0 H2 Store Tank,H2 Store Tank,1000.0,inf,True
ZA0 H2 UHS,H2 UHS,10.0,20000.0,True


In [ ]:
n_flat = n_year.copy()
n_seasonal = n_year.copy()
n_seasonal.generators_t.p_max_pu["ZA0 solar"] = solar_profiles["YEAR_SEASONAL"]
year_cases = {"YEAR_FLAT": n_flat, "YEAR_SEASONAL": n_seasonal}
# Demand, efficiencies, water input, investment options and costs are identical.


In [ ]:
n_flat.optimize(solver_name=SOLVER, threads=1, log_to_console=False);
# Expect daily battery cycling and a small H2 inventory to cover solar-free hours.


INFO:linopy.model: Solve problem using Highs solver


INFO:linopy.model:Solver options:
 - threads: 1
 - log_to_console: False


INFO:linopy.io:Writing objective.


Writing constraints.:   0%|          | 0/23 [00:00<?, ?it/s]

Writing constraints.:  39%|███▉      | 9/23 [00:00<00:00, 89.50it/s]

Writing constraints.:  78%|███████▊  | 18/23 [00:00<00:00, 70.45it/s]

... (3 outputs left out)

Writing continuous variables.: 100%|██████████| 11/11 [00:00<00:00, 101.16it/s]


INFO:linopy.io: Writing time: 0.5s


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 40889 primals, 90534 duals
Objective: 4.40e+06
Solver: highs
Runtime: 19.89s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-ext-p-lower, Generator-ext-p-upper, Link-ext-p-lower, Link-ext-p-upper, Store-ext-e-lower, Store-ext-e-upper, StorageUnit-ext-p_dispatch-lower, StorageUnit-ext-p_dispatch-upper, StorageUnit-ext-p_store-lower, StorageUnit-ext-p_store-upper, StorageUnit-ext-state_of_charge-lower, StorageUnit-ext-state_of_charge-upper, StorageUnit-energy_balance, Store-energy_balance were not assigned to the network.


In [ ]:
n_seasonal.optimize(solver_name=SOLVER, threads=1, log_to_console=False);
# Expect H2 to accumulate over several months and supply export and reconversion
# through the weaker solar season. The battery still shifts energy within a day.


INFO:linopy.model: Solve problem using Highs solver


INFO:linopy.model:Solver options:
 - threads: 1
 - log_to_console: False


INFO:linopy.io:Writing objective.


Writing constraints.:   0%|          | 0/23 [00:00<?, ?it/s]

Writing constraints.:  35%|███▍      | 8/23 [00:00<00:00, 72.93it/s]

Writing constraints.:  70%|██████▉   | 16/23 [00:00<00:00, 59.27it/s]

... (3 outputs left out)

Writing continuous variables.: 100%|██████████| 11/11 [00:00<00:00, 153.03it/s]


INFO:linopy.io: Writing time: 0.54s


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 40889 primals, 90534 duals
Objective: 6.20e+06
Solver: highs
Runtime: 23.39s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-ext-p-lower, Generator-ext-p-upper, Link-ext-p-lower, Link-ext-p-upper, Store-ext-e-lower, Store-ext-e-upper, StorageUnit-ext-p_dispatch-lower, StorageUnit-ext-p_dispatch-upper, StorageUnit-ext-p_store-lower, StorageUnit-ext-p_store-upper, StorageUnit-ext-state_of_charge-lower, StorageUnit-ext-state_of_charge-upper, StorageUnit-energy_balance, Store-energy_balance were not assigned to the network.


In [ ]:
year_summary = pd.DataFrame(index=year_cases)
for name, network in year_cases.items():
    weights = network.snapshot_weightings.generators
    inventory = network.stores_t.e["ZA0 H2 UHS"]
    monthly_inventory = inventory.resample("MS").mean()
    year_summary.loc[name, "solar [MW_el]"] = network.generators.at["ZA0 solar", "p_nom_opt"]
    year_summary.loc[name, "battery [MW_el]"] = network.storage_units.at["ZA0 battery", "p_nom_opt"]
    year_summary.loc[name, "UHS capacity [MWh_H2]"] = network.stores.at["ZA0 H2 UHS", "e_nom_opt"]
    year_summary.loc[name, "monthly mean H2 range [MWh_H2]"] = monthly_inventory.max() - monthly_inventory.min()
    year_summary.loc[name, "H2 export [MWh_H2/year]"] = network.loads_t.p["H2 export load"].mul(weights).sum()
    year_summary.loc[name, "system cost [EUR/year]"] = network.statistics.system_cost(nice_names=False).sum()
year_summary.round(2)

,solar [MW_el],battery [MW_el],UHS capacity [MWh_H2],monthly mean H2 range [MWh_H2],H2 export [MWh_H2/year],system cost [EUR/year]
YEAR_FLAT,47.36,18.55,15.00,0.00,8760.0,4404557.82
YEAR_SEASONAL,61.95,18.55,13471.42,13154.22,8760.0,6203598.37


In [ ]:
# Daily means show inventory levels and the slow cycle, not daily energy production.
h2_inventory = pd.DataFrame({name: network.stores_t.e["ZA0 H2 UHS"]
                             for name, network in year_cases.items()})
fig = h2_inventory.resample("D").mean().div(1000).plot(
    color_discrete_sequence=[n.carriers.at["H2 Store Tank", "color"]] * 2,
    line_dash="variable", line_dash_sequence=["dash", "solid"],
    height=350, labels={"value": "Daily mean hydrogen inventory [GWh_H2]"})
fig.show()

In [ ]:
battery_inventory = pd.DataFrame({name: network.storage_units_t.state_of_charge["ZA0 battery"]
                                  for name, network in year_cases.items()})
fig = battery_inventory.loc["2013-07-01":"2013-07-07"].plot(
    color_discrete_sequence=[n.carriers.at["battery", "color"]] * 2,
    line_dash="variable", line_dash_sequence=["dash", "solid"],
    height=300, labels={"value": "Battery inventory in a July week [MWh_el]"})
fig.show()

The monthly mean H2 inventory distinguishes an inter-season transfer from a
larger daily cycle. Compare it with the battery's repeated daily charging and
discharging. The two cases deliver the same 8,760 MWh_H2/year; their system costs
cover a full year and cannot be compared directly with the 24-hour costs above.

**Try:** increase the underground storage cost or allow wind investment again.
Predict whether the optimum shifts towards more generation or less seasonal
inventory. A zero seasonal-storage investment can also be a meaningful result.

In [ ]:
for name, network in year_cases.items():
    network.meta = {"tutorial": "Synthetic full-year solar and hydrogen comparison",
                    "scenario": name, "step_hours": step_hours,
                    "h2_uhs_annual_cost_EUR_per_MWh": 10.0,
                    "h2_uhs_potential_MWh": 20_000,
                    "source_commit": "d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f"}
    result_dir = work_dir / "results" / name / "postnetworks"
    result_dir.mkdir(parents=True, exist_ok=True)
    network.export_to_netcdf(result_dir / f"elec_s_1_ec_lcopt_{step_hours}h_sector.nc")
year_summary.to_csv(work_dir / f"seasonal-comparison-{step_hours}h.csv")


INFO:pypsa.network.io:Exported network 'Unnamed Network' saved to 'C:\Users\ljansen\projects\pypsa-meets-earth-lab-2026\tmp\final-review-execution\work\results\YEAR_FLAT\postnetworks\elec_s_1_ec_lcopt_3h_sector.nc contains: buses, carriers, links, loads, storage_units, generators, sub_networks, stores


INFO:pypsa.network.io:Exported network 'Unnamed Network' saved to 'C:\Users\ljansen\projects\pypsa-meets-earth-lab-2026\tmp\final-review-execution\work\results\YEAR_SEASONAL\postnetworks\elec_s_1_ec_lcopt_3h_sector.nc contains: buses, carriers, links, loads, storage_units, generators, sub_networks, stores
